In [2]:
import pandas as pd
df = pd.read_excel("../data/online_retail_II.xlsx")
df.head(10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [3]:
returns = df.copy()

In [4]:
returns["Is_Return"] = returns['Quantity'] < 0

In [5]:
return_count = returns["Is_Return"].sum()

print("Returned Transactions:", return_count)

Returned Transactions: 12326


In [6]:
return_rate = (
    return_count / len(returns)
) * 100

print(f"Return Transaction Rate: {return_rate:.2f}%")

Return Transaction Rate: 2.35%


In [7]:
returned_units = returns.loc[
    returns["Is_Return"], "Quantity"
].abs().sum()

print("Total Returned Units:", returned_units)

Total Returned Units: 579547


In [8]:
sold_units = returns.loc[
    ~returns["Is_Return"], "Quantity"
].sum()

print("Total Units Sold:", sold_units)

Total Units Sold: 6011588


In [9]:
unit_return_rate = (
    returned_units / sold_units
) * 100

print(f"Unit Return Rate: {unit_return_rate:.2f}%")

Unit Return Rate: 9.64%


In [10]:
returns["Revenue"] = returns["Quantity"] * returns["Price"]

In [11]:
returned_revenue = returns.loc[
    returns["Is_Return"], "Revenue"
].abs().sum()

print(f"Returned Revenue: {returned_revenue:,.2f}")

Returned Revenue: 630,228.94


In [12]:
sales_revenue = returns.loc[
    ~returns["Is_Return"], "Revenue"
].sum()

print(f"Sales Revenue: {sales_revenue:,.2f}")

Sales Revenue: 10,169,713.57


In [13]:
return_revenue_rate = (
    returned_revenue / sales_revenue
) * 100

print(f"Return Revenue Rate: {return_revenue_rate:.2f}%")

Return Revenue Rate: 6.20%


In [15]:
returned_products = (
    returns[returns["Is_Return"]]
    .groupby(["StockCode", "Description"])["Quantity"]
    .sum()
    .abs()
    .sort_values(ascending=False)
    .head(20)
)

print("Top 10 Products by Returned Units:")
print(returned_products)

Top 10 Products by Returned Units:
StockCode  Description                        
84016      ebay sales                             13500
22126      given away                             10000
20852      given away                             10000
84347      ROTATING SILVER ANGELS T-LIGHT HLDR     9374
16235      ?                                       9000
10120      Zebra invcing error                     9000
20852      Given away                              8800
21088      SET/6 FRUIT SALAD PAPER CUPS            7128
84016      Ebay sales by the box.                  7100
21096      SET/6 FRUIT SALAD  PAPER PLATES         7008
16047      POP ART PEN CASE & PENS                 5184
85110      BLACK SILVER FLOWER T-LIGHT HOLDER      5040
37340      MULTICOLOUR SPRING FLOWER MUG           4993
16046      TEATIME PEN CASE & PENS                 4632
35015      wet damages                             4354
85160A     WHITE BIRD GARDEN DESIGN MUG            4320
22433      my error - 

In [16]:
return_special_terms = (
    returns.loc[
        returns["Is_Return"] &
        returns["Description"].fillna("").str.contains(
            "given away|ebay|error|adjust|postage|manual|amazon|damaged|lost|wrong",
            case=False,
            na=False
        ),
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
    ]
)

print(return_special_terms.head(20))

       Invoice StockCode Description  Quantity    Price  Customer ID
927    C489538      POST     POSTAGE        -1     9.58      15796.0
3053   C489651         M      Manual        -1     5.10      17804.0
3162    489660     35956        lost     -1043     0.00          NaN
3451   C489685      POST     POSTAGE        -1    18.00      12523.0
5897   C489859         M      Manual        -1    69.57          NaN
9103   C490117      POST     POSTAGE        -1     2.99      16570.0
9153   C490120      POST     POSTAGE        -2    18.00      14277.0
9259   C490126         M      Manual        -1     5.95      15884.0
9307   C490129         M      Manual        -1  1998.49      15482.0
9308    490130     21493       lost?      -600     0.00          NaN
11194  C490296      POST     POSTAGE        -1    28.00      12510.0
11553  C490307      POST     POSTAGE        -1     8.97          NaN
17273  C490748         M      Manual        -1   309.73      12748.0
17427   490765     21450     damag

In [17]:
special_return_count = len(return_special_terms)

print("Special Return Transactions:", special_return_count)


Special Return Transactions: 502


In [18]:
special_return_units = return_special_terms["Quantity"].abs().sum()

print("Special Return Units:", special_return_units)

Special Return Units: 73927


In [20]:
special_codes = ["POST", "M", "DOT"]

special_terms = (
    "postage|manual|dotcom|given away|ebay|error|damaged|"
    "damage|lost|adjust|bad debt|voucher"
)


In [21]:
genuine_returns = returns[
    (returns["Is_Return"]) &
    (returns["Price"] > 0) &
    (~returns["StockCode"].isin(special_codes)) &
    (~returns["Description"].fillna("").str.contains(
        special_terms,
        case=False,
        na=False
    ))
].copy()

In [22]:
print("Genuine Return Transactions:", len(genuine_returns))

print(
    "Genuine Returned Units:",
    genuine_returns["Quantity"].abs().sum()
)

Genuine Return Transactions: 9775
Genuine Returned Units: 211844


In [23]:
genuine_return_revenue = (
    genuine_returns["Revenue"].abs().sum()
)

print(
    f"Genuine Return Revenue: "
    f"{genuine_return_revenue:,.2f}"
)

Genuine Return Revenue: 347,603.08


In [24]:
genuine_return_rate = (
    genuine_return_revenue / sales_revenue
) * 100

print(
    f"Genuine Return Revenue Rate: "
    f"{genuine_return_rate:.2f}%"
)

Genuine Return Revenue Rate: 3.42%


In [25]:
top_returned_products = (
    genuine_returns
    .groupby(["StockCode", "Description"])
    .agg(
        Returned_Units=("Quantity", lambda x: x.abs().sum()),
        Return_Revenue=("Revenue", lambda x: x.abs().sum())
    )
    .sort_values("Returned_Units", ascending=False)
    .head(10)
)

print(top_returned_products)

                                               Returned_Units  Return_Revenue
StockCode Description                                                        
84347     ROTATING SILVER ANGELS T-LIGHT HLDR            9374          312.60
21088     SET/6 FRUIT SALAD PAPER CUPS                   7128          570.24
21096     SET/6 FRUIT SALAD  PAPER PLATES                7008          911.04
16047     POP ART PEN CASE & PENS                        5184          414.72
85110     BLACK SILVER FLOWER T-LIGHT HOLDER             5040          387.36
37340     MULTICOLOUR SPRING FLOWER MUG                  4993          500.05
16046     TEATIME PEN CASE & PENS                        4632          389.04
85160A    WHITE BIRD GARDEN DESIGN MUG                   4320          561.60
85184D    S/4 BLUE ROUND DECOUPAGE BOXES                 3940         1664.92
16162L    THE KING GIFT BAG                              3744          187.20


In [26]:
product_returns = (
    returns[
        (~returns["Is_Return"]) &
        (returns["Price"] > 0) &
        (~returns["StockCode"].isin(special_codes))
    ]
    .groupby(["StockCode", "Description"])
    .agg(
        Sold_Units=("Quantity", "sum"),
        Sales_Revenue=("Revenue", "sum")
    )
)

In [27]:
returned_by_product = (
    genuine_returns
    .groupby(["StockCode", "Description"])
    .agg(
        Returned_Units=("Quantity", lambda x: x.abs().sum()),
        Return_Revenue=("Revenue", lambda x: x.abs().sum())
    )
)

In [28]:
product_return_analysis = product_returns.join(
    returned_by_product,
    how="left"
)

In [29]:
product_return_analysis[
    ["Returned_Units", "Return_Revenue"]
] = product_return_analysis[
    ["Returned_Units", "Return_Revenue"]
].fillna(0)

In [30]:
product_return_analysis["Return_Rate"] = (
    product_return_analysis["Returned_Units"]
    / product_return_analysis["Sold_Units"]
) * 100

In [31]:
high_return_products = (
    product_return_analysis[
        product_return_analysis["Sold_Units"] >= 100
    ]
    .sort_values("Return_Rate", ascending=False)
    .head(10)
)

print(high_return_products)

                                        Sold_Units  Sales_Revenue  \
StockCode Description                                               
D         Discount                             196         397.89   
79323W    WHITE CHERRY LIGHTS                  978        5850.06   
79323B    BLACK CHERRY LIGHTS                  235        1518.45   
37471     MULTICOLOUR POLKADOT PLATE           684          68.40   
20715     LITTLE FLOWER SHOPPER BAG            400         424.00   
21392     RED POLKADOT PUDDING BOWL           3686         568.81   
21393     BLUE POLKADOT PUDDING BOWL          2545         429.49   
21749     LARGE RED RETROSPOT WINDMILL         753        1565.81   
72045D    ROSES ON BLUE TEACUP CANDLE          539         918.65   
20661     BLUE POLKADOT PURSE                  372         251.96   

                                        Returned_Units  Return_Revenue  \
StockCode Description                                                    
D         Discount     

In [32]:
returns["Is_Cancelled"] = (
    returns["Invoice"].astype(str).str.startswith("C")
)

In [33]:
print(
    returns.loc[
        returns["Is_Return"],
        "Is_Cancelled"
    ].value_counts()
)

Is_Cancelled
True     10205
False     2121
Name: count, dtype: int64


In [34]:
actual_returns = returns[
    (returns["Is_Return"]) &
    (~returns["Is_Cancelled"]) &
    (returns["Price"] > 0) &
    (~returns["StockCode"].isin(special_codes)) &
    (~returns["Description"].fillna("").str.contains(
        special_terms,
        case=False,
        na=False
    ))
].copy()

In [35]:
print("Actual Return Transactions:", len(actual_returns))
print(
    "Actual Returned Units:",
    actual_returns["Quantity"].abs().sum()
)
print(
    f"Actual Return Revenue: "
    f"{actual_returns['Revenue'].abs().sum():,.2f}"
)

Actual Return Transactions: 0
Actual Returned Units: 0
Actual Return Revenue: 0.00


In [36]:
cancelled = returns[
    returns["Is_Cancelled"]
].copy()

In [37]:
cancelled_transactions = len(cancelled)

print(
    "Cancelled Transactions:",
    cancelled_transactions
)

Cancelled Transactions: 10206


In [38]:
cancellation_rate = (
    cancelled_transactions / len(returns)
) * 100

print(
    f"Cancellation Rate: "
    f"{cancellation_rate:.2f}%"
)

Cancellation Rate: 1.94%


In [39]:
cancelled_units = (
    cancelled["Quantity"]
    .abs()
    .sum()
)

print(
    "Cancelled Units:",
    cancelled_units
)

Cancelled Units: 213420


In [40]:
cancelled_revenue = (
    cancelled["Revenue"]
    .abs()
    .sum()
)

print(
    f"Cancelled Revenue: "
    f"{cancelled_revenue:,.2f}"
)

Cancelled Revenue: 630,602.51


In [41]:
top_cancelled_products = (
    cancelled
    .groupby(["StockCode", "Description"])
    .agg(
        Cancelled_Units=("Quantity", lambda x: x.abs().sum()),
        Cancelled_Revenue=("Revenue", lambda x: x.abs().sum())
    )
    .sort_values("Cancelled_Units", ascending=False)
    .head(10)
)

print(top_cancelled_products)

                                               Cancelled_Units  \
StockCode Description                                            
84347     ROTATING SILVER ANGELS T-LIGHT HLDR             9374   
21088     SET/6 FRUIT SALAD PAPER CUPS                    7128   
21096     SET/6 FRUIT SALAD  PAPER PLATES                 7008   
16047     POP ART PEN CASE & PENS                         5184   
85110     BLACK SILVER FLOWER T-LIGHT HOLDER              5040   
37340     MULTICOLOUR SPRING FLOWER MUG                   4993   
16046     TEATIME PEN CASE & PENS                         4632   
85160A    WHITE BIRD GARDEN DESIGN MUG                    4320   
85184D    S/4 BLUE ROUND DECOUPAGE BOXES                  3940   
16162L    THE KING GIFT BAG                               3744   

                                               Cancelled_Revenue  
StockCode Description                                             
84347     ROTATING SILVER ANGELS T-LIGHT HLDR             312.60  
21088 

In [42]:
product_sales = (
    returns[
        (~returns["Is_Cancelled"]) &
        (returns["Quantity"] > 0) &
        (returns["Price"] > 0)
    ]
    .groupby(["StockCode", "Description"])
    .agg(
        Sold_Units=("Quantity", "sum"),
        Sales_Revenue=("Revenue", "sum")
    )
)

In [43]:
product_cancelled = (
    cancelled
    .groupby(["StockCode", "Description"])
    .agg(
        Cancelled_Units=("Quantity", lambda x: x.abs().sum()),
        Cancelled_Revenue=("Revenue", lambda x: x.abs().sum())
    )
)

In [44]:
product_cancellation_analysis = product_sales.join(
    product_cancelled,
    how="left"
)

In [45]:
product_cancellation_analysis[
    ["Cancelled_Units", "Cancelled_Revenue"]
] = product_cancellation_analysis[
    ["Cancelled_Units", "Cancelled_Revenue"]
].fillna(0)


In [46]:
product_cancellation_analysis["Cancellation_Rate"] = (
    product_cancellation_analysis["Cancelled_Units"]
    / product_cancellation_analysis["Sold_Units"]
) * 100

In [47]:
high_cancellation_products = (
    product_cancellation_analysis[
        product_cancellation_analysis["Sold_Units"] >= 100
    ]
    .sort_values("Cancellation_Rate", ascending=False)
    .head(10)
)

print(high_cancellation_products)

                                        Sold_Units  Sales_Revenue  \
StockCode Description                                               
D         Discount                             196         397.89   
79323W    WHITE CHERRY LIGHTS                  978        5850.06   
79323B    BLACK CHERRY LIGHTS                  235        1518.45   
20715     LITTLE FLOWER SHOPPER BAG            400         424.00   
37471     MULTICOLOUR POLKADOT PLATE           684          68.40   
21392     RED POLKADOT PUDDING BOWL           3686         568.81   
21393     BLUE POLKADOT PUDDING BOWL          2545         429.49   
21749     LARGE RED RETROSPOT WINDMILL         753        1565.81   
72045D    ROSES ON BLUE TEACUP CANDLE          539         918.65   
20661     BLUE POLKADOT PURSE                  372         251.96   

                                        Cancelled_Units  Cancelled_Revenue  \
StockCode Description                                                        
D         Disco

In [48]:
cancelled["Month"] = (
    cancelled["InvoiceDate"]
    .dt.to_period("M")
)

In [49]:
monthly_cancellations = (
    cancelled
    .groupby("Month")
    .agg(
        Cancelled_Transactions=("Invoice", "count"),
        Cancelled_Units=("Quantity", lambda x: x.abs().sum()),
        Cancelled_Revenue=("Revenue", lambda x: x.abs().sum())
    )
    .reset_index()
)

In [50]:
print(monthly_cancellations)

      Month  Cancelled_Transactions  Cancelled_Units  Cancelled_Revenue
0   2009-12                    1015            10018           25838.65
1   2010-01                     686             3880           28675.61
2   2010-02                     576             6954           20995.45
3   2010-03                     844             4494           67721.37
4   2010-04                     626             6483           37354.20
5   2010-05                     983            16614           44536.03
6   2010-06                     793            15494           72483.53
7   2010-07                     734             6185           31444.79
8   2010-08                     573             6702           40498.57
9   2010-09                     812            95320           70682.58
10  2010-10                    1041            11529           81389.69
11  2010-11                    1194            17718           47617.84
12  2010-12                     329            12029           6